# Solutions · Chapter 05-11 · Gradient boosting

Worked answers to every exercise in `notebooks/05_regression/05-11_boosting.ipynb`.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")


def true_curve(x):
    return np.sin(1.2 * x) * 3 + 0.5 * x


NOISE_SD = 1.5
curve_rng = np.random.default_rng(7)
curve_x = curve_rng.uniform(-4, 4, 400)
curve_y = true_curve(curve_x) + curve_rng.normal(0, NOISE_SD, 400)
fit_x, held_x, fit_y, held_y = train_test_split(curve_x, curve_y, test_size=0.5,
                                                random_state=0)


def held_out_rmse(prediction):
    return float(np.sqrt(((held_y - prediction) ** 2).mean()))


print("%d rows to fit, %d held out, noise floor %.2f"
      % (len(fit_x), len(held_x), NOISE_SD))

## Quick understanding

### E1 · What each does with its trees

**A forest fits its trees independently on bootstrap resamples and averages them.** Every tree attempts
the whole problem; the average cancels their disagreements. **It attacks variance.**

**Boosting fits its trees in sequence, each one to the residuals the previous ones left, adding a fraction
of each.** No tree attempts the whole problem. **It attacks bias** - each round removes a piece of the
error the ensemble is still making.

That is why 05-10's forest barely improved on a depth-3 tree for a *smooth* one-dimensional curve (the
bias was already small) while boosting beat the forest by 4.1 RMSE on the shop data, where there was
structure left to extract.

### E2 · Why "more trees" is safe in one and not the other

**In a forest every tree estimates the same quantity** - the conditional mean - from a different
resample. Averaging more independent estimates of one quantity cannot make the average worse; it only
makes it more precise. The chapter measured the improvement from 25 to 300 trees at 0.0277 RMSE, small
but never negative.

**In boosting each tree is fitted to what the previous ones left.** Once the signal has been extracted,
the residuals are noise - and the next tree fits *that*, enthusiastically, because nothing in the
objective distinguishes noise from signal. Held-out RMSE went 1.5179 at round 10 to 1.6097 at round 60.

> **The forest's trees are interchangeable; boosting's are cumulative.** Adding to an average is safe;
> adding to a sum is not.

### E3 · Why fitting the residuals is a gradient step

For squared-error loss on one row, with `F` the current model's prediction:

$$L = \tfrac{1}{2}(y - F)^2, \qquad \frac{\partial L}{\partial F} = -(y - F) = -\,\text{residual}$$

**The negative gradient of the loss with respect to the prediction is exactly the residual.** So "fit a
tree to the residuals and add `rate` times it" is `F <- F - rate * (gradient)` - 05-06's update rule, with
the step taken in the space of functions rather than the space of coefficients.

The practical payoff is that any differentiable loss works. With absolute error the gradient is the
*sign* of the residual, so the trees are fitted to +1 or -1 - which is what makes it robust.

## Hand calculation

### E4 · One round on three rows

Targets 10, 14, 20. The starting prediction is the mean: `44 / 3 = ` **14.6667** for all three.

| target | prediction | residual |
|---|---|---|
| 10 | 14.6667 | **-4.6667** |
| 14 | 14.6667 | **-0.6667** |
| 20 | 14.6667 | **+5.3333** |

Adding `0.5 x residual` to each:

| target | new prediction |
|---|---|
| 10 | `14.6667 - 2.3333 = ` **12.3333** |
| 14 | `14.6667 - 0.3333 = ` **14.3333** |
| 20 | `14.6667 + 2.6667 = ` **17.3333** |

### E5 · Two more rounds

| round | predictions | residuals |
|---|---|---|
| 1 | 12.3333, 14.3333, 17.3333 | -2.3333, -0.3333, +2.6667 |
| 2 | **11.1667, 14.1667, 18.6667** | -1.1667, -0.1667, +1.3333 |
| 3 | **10.5833, 14.0833, 19.3333** | -0.5833, -0.0833, +0.6667 |

**Every residual is exactly half of the previous one, and the predictions converge on the targets.**

The reason is one line: if the tree predicts the residual perfectly, the new residual is
`r - rate * r = (1 - rate) * r`. At `rate = 0.5` that is a halving every round, **geometrically, forever**.

**And this is the whole chapter in miniature.** With three rows and a tree that can fit them exactly, the
model converges on the training targets - which is to say it memorises them. Nothing in the procedure
stops it. On real data the "residual" contains noise as well as signal, and the geometric decay applies
to both.

### E6 · How many rounds at rate 0.1?

Each round multiplies the residual by `1 - 0.1 = 0.9`, so after `k` rounds it is `0.9^k` of its start.

$$0.9^k < 0.1 \quad\Longrightarrow\quad k > \frac{\ln 0.1}{\ln 0.9} = \frac{-2.3026}{-0.1054} = 21.85$$

**22 rounds.**

The general form is worth carrying: reaching a fraction `f` of the starting residual takes
`ln(f) / ln(1 - rate)` rounds, which is **inversely proportional to the rate** for small rates. Ten times
smaller a learning rate is about ten times as many rounds - which is exactly the 8-against-323 the chapter
measured between rates 1.0 and 0.01.

### E7 · Train 0.4, held-out 2.2 at round 800; 1.8 and 1.9 at round 60

**What happened: it overfitted, badly, between rounds 60 and 800.**

At round 60 the gap was 0.1 and both numbers were near each other - a healthy model. By round 800 the
training error had fallen to 0.4 while the held-out error *rose* to 2.2, a gap of 1.8. The extra 740
rounds were spent fitting noise.

**The model to ship is the one from round 60**, whose held-out estimate is 1.9.

**Two caveats worth stating with it.** Round 60 was chosen by looking at this held-out set, so 1.9 is
optimistic - the honest estimate needs a third split, or the nested procedure of 04-07. And the true
optimum may not be exactly 60; the chapter's curves were flat near their minimum, so anything from
roughly 40 to 100 is likely to be within noise of the best.

In [ ]:
fig, (left, middle, right) = plt.subplots(1, 3, figsize=(15, 4.4))

# --- E5: the residuals on three rows, halving every round -------------------
hand_targets = np.array([10.0, 14.0, 20.0])
hand_prediction = np.full(3, hand_targets.mean())
history = [hand_targets - hand_prediction]
for _ in range(4):
    hand_prediction = hand_prediction + 0.5 * (hand_targets - hand_prediction)
    history.append(hand_targets - hand_prediction)
history = np.array(history)

width = 0.26
for row, colour in enumerate(["#c0392b", "#888888", "#1f6f4a"]):
    left.bar(np.arange(len(history)) + (row - 1) * width, history[:, row], width,
             color=colour, label="target %.0f" % hand_targets[row])
left.axhline(0, color="#333333", lw=1.0)
for step in range(1, len(history)):
    left.annotate("", xy=(step - 0.5, history[step, 2]), xytext=(step - 1.5, history[step - 1, 2]),
                  arrowprops=dict(arrowstyle="->", color="#1f6f4a", lw=1.0, ls=":"))
left.text(1.6, 4.4, r"each bar is $\times\,0.5$" "\n" "of the one before",
          fontsize=9, color="#1f6f4a")
left.set_xticks(range(len(history)))
left.set_xlabel("round")
left.set_ylabel("residual still to remove")
left.set_title("E5 - the residuals halve, forever")
left.legend(fontsize=8, loc="lower right")

# --- E6: how far the residual has fallen, by rate ---------------------------
rounds_axis = np.arange(0, 61)
label_height = {0.5: 0.62, 0.3: 0.48, 0.1: 0.34, 0.05: 0.20}
for rate, colour in [(0.5, "#c0392b"), (0.3, "#e08a1e"), (0.1, "#1f6f4a"), (0.05, "#2c5f9e")]:
    remaining = (1 - rate) ** rounds_axis
    middle.plot(rounds_axis, remaining, lw=2.0, color=colour, label="rate %.2f" % rate)
    crossing = int(np.ceil(np.log(0.1) / np.log(1 - rate)))
    middle.scatter([crossing], [(1 - rate) ** crossing], s=55, color=colour, zorder=5)
    middle.annotate("%d rounds" % crossing, xy=(crossing, (1 - rate) ** crossing),
                    xytext=(crossing + 6, label_height[rate]), fontsize=9, color=colour,
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none"),
                    arrowprops=dict(arrowstyle="->", color=colour, lw=1.0))
middle.axhline(0.1, color="#333333", lw=1.0, ls="--")
middle.text(59, 0.125, "10% of the starting residual", fontsize=9, color="#333333", ha="right")
middle.set_ylim(-0.02, 1.02)
middle.set_xlabel("round")
middle.set_ylabel("fraction of the residual left")
middle.set_title(r"E6 - decay is $(1-\mathrm{rate})^k$")
middle.legend(fontsize=8)

# --- E7: the two shapes a pair of curves can have ---------------------------
diagnostic = GradientBoostingRegressor(n_estimators=800, learning_rate=0.1, max_depth=2,
                                       random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
on_fit_curve = np.array([float(np.sqrt(((fit_y - p) ** 2).mean()))
                         for p in diagnostic.staged_predict(fit_x.reshape(-1, 1))])
on_held_curve = np.array([held_out_rmse(p)
                          for p in diagnostic.staged_predict(held_x.reshape(-1, 1))])
best = int(np.argmin(on_held_curve))
right.plot(np.arange(1, 801), on_fit_curve, lw=2.0, color="#2c5f9e", label="training")
right.plot(np.arange(1, 801), on_held_curve, lw=2.0, color="#c0392b", label="held out")
right.axvline(best + 1, color="#1f6f4a", lw=1.4, ls="--")
right.axvspan(best + 1, 800, color="#c0392b", alpha=0.06)
right.annotate("ship this one\n(round %d)" % (best + 1), xy=(best + 1, on_held_curve[best]),
               xytext=(best + 90, on_held_curve[best] + 0.55), fontsize=9, color="#1f6f4a",
               arrowprops=dict(arrowstyle="->", color="#1f6f4a", lw=1.2))
right.annotate("", xy=(760, on_held_curve[759]), xytext=(760, on_fit_curve[759]),
               arrowprops=dict(arrowstyle="<->", color="#333333", lw=1.3))
right.text(600, (on_held_curve[759] + on_fit_curve[759]) / 2, "the gap\nis the warning",
           fontsize=9, color="#333333", ha="right", va="center")
right.set_xlabel("round")
right.set_ylabel("RMSE")
right.set_title("E7 - training down, held out up")
right.legend(fontsize=9)

fig.suptitle("The three hand calculations, drawn", y=1.02)
fig.tight_layout()
plt.show()

print("held-out minimum %.4f at round %d; at round 800 training %.4f, held out %.4f"
      % (on_held_curve[best], best + 1, on_fit_curve[-1], on_held_curve[-1]))

**Left: E5's arithmetic, as a shape.** The three bars shrink towards zero by the same factor every round
and never quite arrive - which is what "geometric" looks like. Notice that the *largest* residual stays
the largest: boosting does not reorder the rows, it scales the whole error vector down.

**Middle: E6, and why the learning rate is a time budget.** The four curves are the same curve stretched.
Read off where each crosses the dashed line - 4, 7, 22, 45 rounds - and the inverse proportionality is
visible without the algebra: halve the rate, roughly double the rounds.

**Right: E7's diagnosis drawn on real curves.** The exercise gave you four numbers from an imaginary run;
this is the same shape measured on the chapter's own data, so the round numbers differ and the diagnosis
does not. The blue line keeps falling because it always can; the red line turns. The green marker is the only round worth shipping, and
everything in the shaded region is the model learning the training set's noise by heart. **The gap between
the two lines at round 800 is not a bug in the fit - it is the model doing exactly what it was asked.**

## Coding

### E8 · A general boosting loop

In [ ]:
def boost(depth, rate, rounds, subsample=1.0, seed=0):
    rng = np.random.default_rng(seed)
    on_fit = np.full(len(fit_y), fit_y.mean())
    on_held = np.full(len(held_y), fit_y.mean())
    curve = []
    for _ in range(rounds):
        residual = fit_y - on_fit
        if subsample < 1.0:
            rows = rng.choice(len(fit_y), int(subsample * len(fit_y)), replace=False)
        else:
            rows = np.arange(len(fit_y))
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(
            fit_x[rows].reshape(-1, 1), residual[rows])
        on_fit = on_fit + rate * tree.predict(fit_x.reshape(-1, 1))
        on_held = on_held + rate * tree.predict(held_x.reshape(-1, 1))
        curve.append(held_out_rmse(on_held))
    return np.array(curve)


checks = []
for depth, rate, rounds in [(1, 0.5, 40), (2, 0.1, 120), (3, 0.05, 200)]:
    mine = boost(depth, rate, rounds)[-1]
    theirs = held_out_rmse(
        GradientBoostingRegressor(n_estimators=rounds, learning_rate=rate,
                                  max_depth=depth, random_state=0).fit(
            fit_x.reshape(-1, 1), fit_y).predict(held_x.reshape(-1, 1)))
    checks.append({"depth": depth, "rate": rate, "rounds": rounds,
                   "my loop": mine, "sklearn": theirs, "difference": abs(mine - theirs)})
print(pd.DataFrame(checks).to_string(index=False, float_format=lambda v: "%.6f" % v))

**Identical on all three settings, to six decimal places.**

The loop is genuinely the algorithm - not a simplification of it. What `GradientBoostingRegressor` adds
beyond these eight lines is other loss functions, subsampling, a line search on the leaf values, and
early stopping.

**One detail that matters and is easy to get wrong:** the running prediction must be updated for **both**
the fitting and the held-out rows inside the loop, using the *same* tree. Recomputing the held-out
predictions at the end from a stored list of trees gives the same answer but is much slower; recomputing
them from a *refitted* model does not, because the trees would differ.

### E9 · Stochastic gradient boosting

In [ ]:
comparison = {}
for fraction in [1.0, 0.6]:
    curve = boost(depth=2, rate=0.1, rounds=400, subsample=fraction)
    comparison[fraction] = curve
    print("subsample %.1f: best round %3d, best RMSE %.4f, RMSE at round 400 %.4f"
          % (fraction, int(np.argmin(curve)) + 1, curve.min(), curve[-1]))

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.6))
for fraction, colour in [(1.0, "#D55E00"), (0.6, "#0072B2")]:
    curve = comparison[fraction]
    ax.plot(np.arange(1, len(curve) + 1), curve, color=colour, linewidth=2.2,
            label="subsample %.1f (best %.4f)" % (fraction, curve.min()))
    ax.plot([int(np.argmin(curve)) + 1], [curve.min()], "o", color=colour, markersize=9)
ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.6,
           label="noise floor, %.2f" % NOISE_SD)
ax.set_xscale("log")
ax.set_ylim(1.42, 1.9)
ax.set_xlabel("boosting round (log scale)")
ax.set_ylabel("held-out RMSE")
ax.set_title("Fitting each tree on 60% of the rows is better, and degrades more slowly",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Subsampling wins on both counts: best RMSE 1.4753 against 1.5166, and 1.6995 against 1.7576 after 400
rounds.**

**The name.** It is called *stochastic* gradient boosting by direct analogy with 05-06's stochastic
gradient descent: each step is computed on a random subset of the rows, so the direction is a noisy
estimate of the true gradient rather than the exact one.

**Why the noise helps here, when it merely saved time in 05-06.** Two mechanisms:

1. **It decorrelates the trees**, which is 05-10's `max_features` argument applied to rows instead of
   columns. Successive trees see different data and make different mistakes.
2. **It stops any single row driving the fit.** A row with a large residual is in only 60% of the rounds,
   so its influence is diluted - a mild version of the robustness the absolute-error loss buys.

**And it is nearly free**, since each tree is fitted on 60% of the data. This is why `subsample=0.8` is a
common default in production boosting libraries and why it is worth trying before anything more elaborate.

### E10 · Depth against the best round

In [ ]:
depth_rows = []
for depth in [1, 2, 3, 5]:
    model = GradientBoostingRegressor(n_estimators=800, learning_rate=0.05,
                                      max_depth=depth, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    per_round = np.array([held_out_rmse(prediction)
                          for prediction in model.staged_predict(held_x.reshape(-1, 1))])
    depth_rows.append({"max_depth": depth, "leaves per tree (max)": 2 ** depth,
                       "best round": int(np.argmin(per_round)) + 1,
                       "best held-out RMSE": per_round.min()})
print(pd.DataFrame(depth_rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

**The best round collapses as depth grows - 418, 71, 56, 56 - and the best RMSE is optimal in the
middle.**

**The pattern in the rounds is a trade between the two capacity knobs.** A depth-1 stump makes one split,
so it removes one piece of structure per round; a depth-3 tree makes up to seven and removes
correspondingly more, so it needs far fewer rounds to arrive. **Depth and round count buy the same thing,
and you may pay in either currency.**

The exchange rate is not clean, though, and it is worth seeing that rather than memorising a formula. If
`rounds x leaves per tree` were conserved the four products would match; they are 836, 284, 448 and 1792,
which agree only to within a factor of six. **The direction is reliable, the arithmetic is not** - a deeper
tree places its extra leaves adaptively, so it removes more per round than a leaf count predicts, and past
depth 3 it is also spending capacity on noise.

**The pattern in the scores is 05-07's capacity curve, again.** Depth 2 wins at 1.5221; depth 1 is
slightly worse (1.5295) because a stump cannot represent an interaction at all - each tree can only ever
say something about one variable; depth 5 is clearly worse (1.6739) because each individual tree already
overfits before the ensemble gets a say.

**The practical rule that follows: depth 2 to 4, and let the round count take the strain.** Boosting works
by accumulating many weak corrections, and a deep tree is not weak.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.4))

capacity = []
for depth, colour in [(1, "#2c5f9e"), (2, "#1f6f4a"), (3, "#e08a1e"), (5, "#c0392b")]:
    model = GradientBoostingRegressor(n_estimators=800, learning_rate=0.05, max_depth=depth,
                                      random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
    per_round = np.array([held_out_rmse(p)
                          for p in model.staged_predict(held_x.reshape(-1, 1))])
    best = int(np.argmin(per_round))
    left.plot(np.arange(1, 801), per_round, lw=1.9, color=colour, label="depth %d" % depth)
    left.scatter([best + 1], [per_round[best]], s=60, color=colour, zorder=5,
                 edgecolor="white", linewidth=1.2)
    capacity.append({"depth": depth, "best round": best + 1, "leaves": 2 ** depth,
                     "colour": colour, "best": per_round[best]})

left.axhline(NOISE_SD, color="#333333", lw=1.0, ls=":")
left.text(9, NOISE_SD + 0.03, "noise floor", fontsize=9, color="#333333")
left.set_xscale("log")
left.set_xticks([1, 10, 100, 800])
left.set_xticklabels(["1", "10", "100", "800"])
left.xaxis.set_minor_formatter(plt.NullFormatter())
left.set_ylim(1.45, 2.6)
left.set_xlabel("round (log scale)")
left.set_ylabel("held-out RMSE")
left.set_title("deeper trees arrive sooner, and leave sooner")
left.legend(fontsize=9)

positions = np.arange(len(capacity))
scores = [row["best"] for row in capacity]
right.bar(positions, scores, 0.55, color=[row["colour"] for row in capacity])
for position, row in zip(positions, capacity):
    right.text(position, row["best"] + 0.012, "%.4f at round %d" % (row["best"], row["best round"]),
               ha="center", fontsize=8.5)
right.axhline(NOISE_SD, color="#333333", lw=1.2, ls=":")
right.text(2.5, NOISE_SD, "noise floor", fontsize=9, color="#333333", ha="center", va="center",
           bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none"), zorder=5)
right.set_xlim(-0.62, 3.55)
right.set_xticks(positions)
right.set_xticklabels(["depth %d" % row["depth"] for row in capacity])
right.set_ylim(1.45, max(scores) * 1.09)
right.set_ylabel("best held-out RMSE")
right.set_title("and the best of them is in the middle")

fig.suptitle("E10 - depth and round count buy the same thing", y=1.02)
fig.tight_layout()
plt.show()

**Left: the minima march leftwards as the trees get deeper**, and the depth-5 curve turns upwards while
the others are still falling. A depth-5 tree is not a weak learner, so the ensemble reaches the noise
floor's neighbourhood in a few dozen rounds and then starts memorising.

**Right: 05-07's capacity curve, one more time.** Depth 1 is short of the floor because a stump cannot
represent an interaction at all; depth 5 is well above it because each tree overfits before the ensemble
gets a say; depth 2 sits closest. **The U is the same U as every other capacity knob in this module** - only
the knob has changed.

### E11 · The histogram implementation

In [ ]:
# SYNTHETIC: 50,000 rows, 8 features, one non-linearity and one interaction
big_rng = np.random.default_rng(5)
n_big = 50_000
big_X = big_rng.normal(size=(n_big, 8))
big_y = (big_X[:, 0] * 2 + np.sin(big_X[:, 1] * 2) * 3 + big_X[:, 2] * big_X[:, 3]
         + big_rng.normal(0, 1, n_big))
big_train, big_test, big_train_y, big_test_y = train_test_split(
    big_X, big_y, test_size=0.3, random_state=0)

timings = []
for label, model in [
        ("GradientBoostingRegressor",
         GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.1,
                                   random_state=0)),
        ("HistGradientBoostingRegressor",
         HistGradientBoostingRegressor(max_iter=200, max_depth=3, learning_rate=0.1,
                                       early_stopping=False, random_state=0))]:
    started = time.time()
    model.fit(big_train, big_train_y)
    elapsed = time.time() - started
    timings.append({"implementation": label, "fit seconds": elapsed,
                    "held-out RMSE": float(np.sqrt(((big_test_y - model.predict(big_test)) ** 2).mean()))})
print(pd.DataFrame(timings).to_string(index=False, float_format=lambda v: "%.4f" % v))
speedup = timings[0]["fit seconds"] / timings[1]["fit seconds"]
print("\nthe histogram version is %.0f times faster, and scores better" % speedup)

**Between thirty and forty times faster on this machine, and slightly more accurate: held-out RMSE 1.0765
against 1.1926.** (The RMSEs are deterministic; the timings will differ on your hardware, so read the
ratio the cell prints rather than the seconds.)

**In one sentence: it stops looking at every distinct value.** The exact implementation sorts each feature
and evaluates every candidate threshold - the exhaustive search of 05-10, which costs `O(n log n)` per
feature per node. The histogram version bins each feature into 255 buckets once, up front, and then
considers only 254 thresholds however many rows there are. The cost per split stops growing with `n`.

**And the accuracy difference is not noise, it is regularisation.** Binning is a coarsening: it forbids
splits that separate values within the same bucket, which are exactly the splits most likely to be
fitting a single row. **The approximation is a feature.**

This is the core idea behind LightGBM and XGBoost's `hist` mode, and it is why they replaced the exact
implementations everywhere. **On 50,000 rows the choice is obvious; below about 10,000 the exact version
is fine and the difference does not matter.**

### E12 · Early stopping, written out

In [ ]:
def boost_with_early_stopping(depth, rate, max_rounds, patience=20,
                              validation_fraction=0.2, seed=0):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(fit_y))
    cut = int(len(fit_y) * (1 - validation_fraction))
    inner, watch = order[:cut], order[cut:]

    on_inner = np.full(len(inner), fit_y[inner].mean())
    on_watch = np.full(len(watch), fit_y[inner].mean())
    on_held = np.full(len(held_y), fit_y[inner].mean())

    best_score, best_round, best_held = np.inf, 0, on_held.copy()
    since_improvement = 0
    for step in range(1, max_rounds + 1):
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(
            fit_x[inner].reshape(-1, 1), fit_y[inner] - on_inner)
        on_inner = on_inner + rate * tree.predict(fit_x[inner].reshape(-1, 1))
        on_watch = on_watch + rate * tree.predict(fit_x[watch].reshape(-1, 1))
        on_held = on_held + rate * tree.predict(held_x.reshape(-1, 1))

        score = float(np.sqrt(((fit_y[watch] - on_watch) ** 2).mean()))
        if score < best_score - 1e-9:
            best_score, best_round, best_held = score, step, on_held.copy()
            since_improvement = 0
        else:
            since_improvement += 1
            if since_improvement >= patience:
                break
    return best_round, best_score, held_out_rmse(best_held), step


chosen, watch_score, held_score, stopped = boost_with_early_stopping(2, 0.05, 800)
print("stopped after %d rounds; best watch score at round %d" % (stopped, chosen))
print("score on the watched slice : %.4f" % watch_score)
print("score on the true held-out : %.4f\n" % held_score)

reference = GradientBoostingRegressor(n_estimators=800, learning_rate=0.05, max_depth=2,
                                      random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
per_round = np.array([held_out_rmse(p)
                      for p in reference.staged_predict(held_x.reshape(-1, 1))])
print("the true optimum, found by looking at the held-out set: round %d, RMSE %.4f"
      % (int(np.argmin(per_round)) + 1, per_round.min()))
print("rounds apart: %d" % abs(chosen - (int(np.argmin(per_round)) + 1)))

**It lands near the optimum, and the gap between "near" and "at" is the honest content of this
exercise.**

The procedure never touches the held-out set - it watches an inner slice carved from the training rows -
so the number it reports is legitimate. It stops some rounds away from the round a held-out sweep would
pick, for two reasons that are both features rather than bugs:

**It trained on 20% fewer rows**, so its trees are slightly different from the reference model's. The
comparison is not quite like for like, which is 04-07's nested-validation observation again.

**The patience means it overshoots by design.** It keeps going for 20 rounds after the last improvement,
then rewinds to the best one - so the *stopping* round and the *chosen* round differ by at least the
patience.

**And it does not matter much**, because the curve is flat near its floor. That flatness is what the
chapter's early-stopping figure was showing, and it is why this crude procedure is the one everyone uses.

**One implementation detail worth copying: keep the best predictions, not just the best score.** A loop
that records only the round number has to refit to recover the model. Storing `best_held` costs one array
copy and gives the answer directly.

## Interpretation

### E13 · Still falling at 5,000 rounds

**What it tells you: the learning rate is too small for the budget, and the model is nowhere near its
capacity.** Nothing is wrong - the model is simply still descending, and every round is still buying
signal.

**Two things to check first, since both are common and neither needs a change:**

- **Is it actually falling, or flat?** Plot the last 1,000 rounds alone. A curve that has improved by
  0.001 over 1,000 rounds is finished; "still falling" and "not yet flat" look identical on a full-range
  plot.
- **Is the improvement worth anything?** Compare the remaining slope with the noise floor and with what
  the business needs.

**What I would change, in order:**

1. **Raise the learning rate.** From 0.01 to 0.05 is five times fewer rounds for essentially the same
   optimum - the chapter measured best RMSE of 1.5199 and 1.5221 at those two rates.
2. **Deepen the trees.** E10 showed depth 3 reaching its optimum at round 56 where depth 1 needed 418.
   Same total capacity, far fewer rounds.
3. **Only then, run longer** - and if you do, use early stopping so the run terminates itself.

**What I would not conclude is that the model is underfitting.** A slowly descending curve is what a small
learning rate looks like, by construction.

### E14 · Boosting beats linear by 30% and the forest by 2%

**What I conclude: the data has real non-linearity or interactions, and boosting has extracted nearly all
of what tree-based models can get.**

The 30% over linear is the structure that a weighted sum cannot represent. The **2%** over the forest is
the informative number: both are tree ensembles with access to the same structure, so a small gap means
boosting's sequential refinement found little the forest's averaging had not. **The remaining error is
mostly noise, or mostly something neither can see.**

**What I would try next, in order of expected value:**

1. **Find out what shape the truth is.** Look at boosting's partial dependence and its top interactions,
   then hand the linear model the two or three terms that suggests. If the linear model catches up, ship
   it - 05-07's shop data ended with a four-column linear model beating everything.
2. **Estimate the floor.** If the forest and boosting agree to 2%, that is weak evidence they have both
   hit the data's limit. 05-08's learning curve settles it: if the curves have met, more rows and more
   tuning are both wasted.
3. **Look for a missing feature**, which is what a floor above the noise means.

**What I would not do is spend a week tuning boosting for the next 1%.** A 2% gap over an untuned forest
says the tuning surface is flat.

## Debugging

### E15 · Brilliant in cross-validation, poor in production

**Three causes, and the last two are specific to what this chapter and 05-10 built:**

**1. Leakage - the general cause.** A feature carrying information unavailable at prediction time is
present in every fold, so cross-validation cannot see it. 04-05 covers this and it remains the most
common single explanation.

**2. Extrapolation - specific to trees.** Boosted models are made of trees, so 05-10's flat line past the
edge of the training range applies unchanged. Any feature that drifts - a date, a counter, a price, a
volume - is frozen at whatever the last training rows said. Cross-validation cannot detect this because
every fold is drawn from the same range. **The check is a chronological split, and a comparison of each
feature's production range against its training range.**

**3. The stopping round was chosen on the same folds it was scored on - specific to boosting.** The number
of rounds is a hyperparameter with an unusually sharp optimum, and choosing it by looking at the
cross-validated curve makes that curve optimistic. 04-07's nested procedure is the fix; the symptom is a
model that is exactly as good as advertised on the data used to tune it and worse everywhere else.

**A fourth if the target is continuous and skewed:** the training loss. If squared error was used and the
target has a long tail, a few extreme training rows may have shaped the model - the chapter measured five
corrupted rows in two hundred taking RMSE from 1.5522 to 5.4986.

### E16 · Four hours to train

**In order, cheapest and highest-yield first:**

1. **Switch to `HistGradientBoostingRegressor`** (or LightGBM/XGBoost with `hist`). E11 measured a
   **40x** speedup on 50,000 rows, *with a better score*. This alone usually ends the problem.
2. **Turn on early stopping.** If the model converges at round 300 and you are running 3,000, you are
   spending 90% of the time making it worse.
3. **Reduce the rounds by raising the learning rate.** 0.01 to 0.1 is roughly ten times fewer rounds for
   a comparable optimum.
4. **Subsample the rows** - `subsample=0.6` was both faster and *better* in E9.
5. **Reduce the data.** Fit on a stratified 20% sample while you are iterating on features and settings,
   and only train the final model on everything. Most of the four hours is usually spent on runs whose
   result you were going to discard.

**And one thing to check before any of that: how many features are being one-hot encoded.** A single
high-cardinality categorical column can turn eight features into eight thousand, and the split search is
linear in that count. Native categorical support - which `HistGradientBoostingRegressor` has - avoids it
entirely.

## Exam and interview reasoning

### E17 · "What is the difference between bagging and boosting?"

> "Both build ensembles of trees, and they do opposite things with them. Bagging - a random forest - fits
> the trees independently on bootstrap resamples and averages them, so it attacks variance: each tree
> tries to solve the whole problem and the disagreements cancel. Boosting fits the trees in sequence, each
> one to the residuals the previous ones left, so it attacks bias: each tree only has to remove a piece of
> the error that is still there.
>
> The practical consequence is the one that matters. Adding trees to a forest can never hurt, so
> `n_estimators` is not a hyperparameter. Adding trees to a boosted model will eventually overfit, so the
> number of rounds is the most important setting there is - and that is what early stopping exists for."

**"Which would you use on 500 rows, and why?"**

> "I would start with the forest and probably ship it. With 500 rows, variance is the binding constraint -
> 05-08's decomposition - and that is precisely what bagging reduces. Boosting reduces bias, which is not
> what is hurting me, and it introduces a sharp hyperparameter that I have to tune on a validation split
> I can barely afford to carve out of 500 rows.
>
> I would still *try* boosting, with a small learning rate, shallow trees and nested cross-validation for
> the round count. But the forest's advantage on small data is that it has almost nothing to get wrong."

**What is being tested:** the first answer should reach "variance versus bias" and the `n_estimators`
asymmetry unprompted. The follow-up checks whether you can connect a model choice to the decomposition
rather than reciting that boosting usually wins.

In [ ]:
def box(ax, cx, cy, w, h, label, colour, fontsize=9):
    ax.add_patch(plt.Rectangle((cx - w / 2, cy - h / 2), w, h, facecolor="white",
                               edgecolor=colour, linewidth=1.6, zorder=3))
    ax.text(cx, cy, label, ha="center", va="center", fontsize=fontsize, color=colour, zorder=4)


def arrow(ax, start, end, colour, style="->", ls="-"):
    ax.annotate("", xy=end, xytext=start, zorder=2,
                arrowprops=dict(arrowstyle=style, color=colour, lw=1.3, linestyle=ls))


fig, (left, right) = plt.subplots(1, 2, figsize=(13.5, 5.0))
for ax in (left, right):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

FOREST, BOOST, GREY = "#2c5f9e", "#1f6f4a", "#666666"

# ---- bagging: independent, parallel, averaged -----------------------------
box(left, 0.5, 0.90, 0.52, 0.11, "training data", GREY)
for cx in [0.155, 0.385, 0.615, 0.845]:
    box(left, cx, 0.55, 0.19, 0.13, "tree", FOREST)
    arrow(left, (0.5, 0.845), (cx, 0.617), FOREST)
    arrow(left, (cx, 0.483), (0.5, 0.262), FOREST)
box(left, 0.5, 0.20, 0.56, 0.12, "average them", FOREST)
left.text(0.5, 0.735, "a bootstrap resample each", ha="center", fontsize=8.5, color=GREY,
          bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none"), zorder=5)
left.text(0.5, 0.045, "every tree attempts the whole problem;\n"
                      "their disagreements cancel  ->  attacks VARIANCE",
          ha="center", fontsize=9.5, color=FOREST)
left.set_title("Bagging (a random forest)", fontsize=12, color=FOREST)

# ---- boosting: sequential, cumulative, summed -----------------------------
box(right, 0.5, 0.90, 0.52, 0.11, "training data", GREY)
centres = [0.14, 0.44, 0.74]
for index, cx in enumerate(centres, start=1):
    box(right, cx, 0.55, 0.20, 0.13, "tree %d" % index, BOOST)
    arrow(right, (cx, 0.483), (0.5, 0.262), BOOST)
arrow(right, (0.5, 0.845), (0.14, 0.617), BOOST)
for gap_left, gap_right in [(0.24, 0.34), (0.54, 0.64)]:
    arrow(right, (gap_left, 0.55), (gap_right, 0.55), BOOST)
    right.text((gap_left + gap_right) / 2, 0.645, "residuals", ha="center", fontsize=7.5,
               color=BOOST, zorder=5,
               bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none"))
right.text(0.92, 0.55, "...", ha="center", va="center", fontsize=14, color=BOOST)
box(right, 0.5, 0.20, 0.56, 0.12, "add a fraction of each", BOOST)
right.text(0.5, 0.045, "no tree attempts the whole problem;\n"
                       "the corrections accumulate  ->  attacks BIAS",
           ha="center", fontsize=9.5, color=BOOST)
right.set_title("Boosting", fontsize=12, color=BOOST)

fig.suptitle("E17 - parallel and interchangeable, against sequential and cumulative", y=0.99)
fig.tight_layout()
plt.show()

**The one structural difference is the direction of the arrows in the middle row**, and every practical
consequence follows from it.

On the left the trees never speak to each other, so the order does not matter, so a tree can be added or
removed without disturbing the rest - which is exactly why `n_estimators` is not a hyperparameter and why
a forest parallelises across cores for free.

On the right each tree exists only because of what the ones before it left behind. **Remove tree 12 and
trees 13 onwards are answering a question that no longer exists.** That is the same fact seen three ways:
the rounds must be tuned, the fit is inherently sequential, and stopping late is a real failure mode
rather than wasted electricity.

> A useful one-line test in an interview: *can you shuffle the trees?* Forest, yes. Boosting, no.

## Transfer to a different situation

### E18 · Insurance claims: mostly zero, a few enormous, underestimation is expensive

**Three separate decisions, and the loss is the one that matters most.**

**The loss.** Squared error is wrong here in both directions. It is dominated by the few enormous claims -
the chapter measured five extreme rows out of two hundred taking RMSE from 1.55 to 5.50 - and it is
symmetric, while the business has explicitly said underestimating is worse than overestimating.

- **Match the loss to the cost asymmetry: a pinball (quantile) loss.** Predicting the 80th percentile
  rather than the mean makes underestimation expensive by construction, which is the stated requirement.
  `HistGradientBoostingRegressor(loss="quantile", quantile=0.8)` does exactly this.
- **If the mean is genuinely wanted, use a distribution that matches the shape.** Claim amounts are
  non-negative and heavily right-skewed; `loss="poisson"`, or modelling `log(1 + amount)`, respects that
  where squared error on the raw amount does not.

**The model.** A two-part structure is standard and worth the extra complexity here: **one model for
whether a claim occurs at all, and a second for its size given that it occurred.** Most rows being zero
means a single model spends most of its capacity on the zeros and is dragged towards them; splitting the
question lets each half be trained on what it is actually predicting.

**The evaluation.** RMSE on the whole population will be dominated by the large claims and will tell you
almost nothing about the routine ones.

- **Report by segment** - error on zero claims, small claims, and large claims separately.
- **Report the quantity the business acts on**, which for insurance is usually total reserve across a
  portfolio, not per-claim accuracy. A model can be poor per claim and excellent in aggregate.
- **Report the underestimation rate explicitly**, since that is the stated cost and no symmetric metric
  contains it.

**And the judgement being tested:** the temptation is to answer "gradient boosting with tuned
hyperparameters". The problem is not the algorithm; it is that the default loss, the default population
and the default metric are all wrong for the question.

In [ ]:
fig, (left, middle, right) = plt.subplots(1, 3, figsize=(15, 4.3))
resid_axis = np.linspace(-4, 4, 400)


def pinball(residual, quantile):
    return np.where(residual >= 0, quantile * residual, (quantile - 1) * residual)


for quantile, colour in [(0.5, "#666666"), (0.8, "#c0392b")]:
    left.plot(resid_axis, pinball(resid_axis, quantile), lw=2.2, color=colour,
              label="quantile %.1f" % quantile)
    middle.plot(resid_axis, np.where(resid_axis >= 0, quantile, quantile - 1), lw=2.2,
                color=colour, drawstyle="steps-mid", label="quantile %.1f" % quantile)
for ax in (left, middle):
    ax.axvline(0, color="#999999", lw=0.8)
    ax.axhline(0, color="#999999", lw=0.8)
    ax.set_xlabel("residual $r = y - F$   (negative = we predicted too high)")
left.set_ylabel("loss")
left.set_title("pinball loss: the V is lopsided")
left.legend(fontsize=9)
left.annotate("under-predicting\ncosts 4x as much", xy=(2.4, pinball(2.4, 0.8)),
              xytext=(-3.8, 2.3), fontsize=9, color="#c0392b",
              arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2))
middle.set_ylabel("target handed to the next tree")
middle.set_ylim(-1.1, 1.1)
middle.set_title("its negative gradient: +0.8 or -0.2")
middle.legend(fontsize=9, loc="lower right")

# SYNTHETIC: 20,000 policies, 78% with no claim, the rest lognormal - shape only,
# not calibrated to any real insurance book
claim_rng = np.random.default_rng(11)
has_claim = claim_rng.random(20_000) < 0.22
claims = np.where(has_claim, claim_rng.lognormal(mean=7.4, sigma=1.3, size=20_000), 0.0)
right.hist(np.clip(claims, 0, 6_000), bins=60, color="#b0b0b0")
right.axvline(claims.mean(), color="#2c5f9e", lw=2.0,
              label="mean %.0f (what squared error targets)" % claims.mean())
right.axvline(np.quantile(claims, 0.8), color="#c0392b", lw=2.0,
              label="80th percentile %.0f" % np.quantile(claims, 0.8))
right.axvline(np.quantile(claims, 0.95), color="#1f6f4a", lw=2.0,
              label="95th percentile %.0f" % np.quantile(claims, 0.95))
right.set_yscale("log")
right.set_xlabel("claim amount (clipped at 6,000 for the plot)")
right.set_ylabel("policies (log scale)")
right.set_title("SYNTHETIC claims: a spike at zero, a long tail")
right.legend(fontsize=8)

fig.suptitle("E18 - the loss, the population and the metric all have to be chosen", y=1.01)
fig.tight_layout()
plt.show()

print("share with no claim  %.3f" % (claims == 0).mean())
print("mean %.0f   median %.0f   80th pct %.0f   95th pct %.0f   largest %.0f"
      % (claims.mean(), np.median(claims), np.quantile(claims, 0.8),
         np.quantile(claims, 0.95), claims.max()))

**Left and middle: the asymmetry is one number in one line of the algorithm.** E3's rule still applies -
the tree is handed the negative gradient - and for the pinball loss that gradient is `+0.8` when we are
too low and `-0.2` when we are too high. A model trained on it is not "biased upwards" by a fudge factor;
it is optimally predicting the 80th percentile, which is what the business asked for when it said
underestimation costs more. **At quantile 0.5 the same picture is symmetric, and pinball loss is exactly
absolute error - a median regressor.**

**Right: why the population needs splitting - and a trap in the answer above.** 77.6% of these policies
claim nothing, the median is 0, the mean is 860 and the largest claim is 188,132. The blue line is the
quantity squared error aims at, and it is a value almost no policy has: *correct* as a portfolio average,
*useless* as a per-policy prediction. That is the practical case for modelling "does a claim occur" and
"how big is it" separately.

**Now read the two percentile lines.** The 80th percentile is 317 - **below** the mean of 860 - because
four rows in five are zero or near it. So `quantile=0.8` on a zero-inflated target is not the conservative
prediction it sounds like; you have to go to the 95th percentile, 4,537, before the prediction is
meaningfully above the mean. The asymmetric *loss* is still the right tool for an asymmetric cost, but
**the quantile has to be chosen against the shape of this distribution, not against the intuition that
0.8 is "high".**

**And the metric follows the same logic.** A single RMSE over this histogram is dominated by the handful
of policies in the right tail; segmenting it - zero, small, large - is the only way to see whether the
model is any good at the 77.6% of rows that are zero.

## Explain it to someone non-technical

### E19 · Boosting, and why trying harder makes it worse

> Imagine a team estimating house prices. The first person gives a rough guess for every house. The second
> does not start again - they look only at where the first was wrong and suggest a small correction. The
> third corrects what is still wrong after that, and so on. Each person handles less than the last, and
> the estimates get steadily better.
>
> The catch is that eventually there is nothing real left to correct, and the next person starts
> explaining away flukes - "this one sold cheap because it rained that day". Keep going and the team gets
> worse. So we stop them as soon as they stop improving on houses they have not seen.

*(105 words.)* The "it rained that day" line is doing the work: it makes overfitting concrete and makes
early stopping sound like the obvious response rather than a technicality.

## Going further

### E20 · Boosting with absolute-error loss

E3 established the rule: **the tree is fitted to the negative gradient of the loss.** For squared error
that is the residual. For absolute error, `L = |y - F|`, the derivative with respect to `F` is `-1` when
the residual is positive and `+1` when it is negative, so the negative gradient is

$$-\frac{\partial L}{\partial F} = \operatorname{sign}(y - F)$$

**The tree is fitted to +1 and -1, and nothing else.** A row that is out by 40 and a row that is out by
0.4 send the identical instruction: *go up*. That single change is the whole of the robustness.

In [ ]:
def boost_absolute(target, rate, rounds, depth=2, median_leaves=False):
    on_fit = np.full(len(target), np.median(target))
    on_held = np.full(len(held_y), np.median(target))
    for _ in range(rounds):
        residual = target - on_fit
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(
            fit_x.reshape(-1, 1), np.sign(residual))          # <- the only change
        if median_leaves:
            # the refinement sklearn makes: replace each leaf's value by the median
            # of the residuals that landed in it, which is the minimiser of |.| there
            leaf_of_fit = tree.apply(fit_x.reshape(-1, 1))
            value = {leaf: float(np.median(residual[leaf_of_fit == leaf]))
                     for leaf in np.unique(leaf_of_fit)}
            step_fit = np.array([value[leaf] for leaf in leaf_of_fit])
            step_held = np.array([value[leaf] for leaf in tree.apply(held_x.reshape(-1, 1))])
        else:
            step_fit = tree.predict(fit_x.reshape(-1, 1))
            step_held = tree.predict(held_x.reshape(-1, 1))
        on_fit = on_fit + rate * step_fit
        on_held = on_held + rate * step_held
    return on_fit, on_held


# SYNTHETIC corruption: 5 of the 200 fitting targets pushed up by 40, as in the chapter
corrupted_y = fit_y.copy()
damaged = np.random.default_rng(3).choice(len(corrupted_y), 5, replace=False)
corrupted_y[damaged] += 40.0

_, plain_sign = boost_absolute(corrupted_y, rate=0.05, rounds=200)
_, median_sign = boost_absolute(corrupted_y, rate=0.05, rounds=200, median_leaves=True)
their_absolute = GradientBoostingRegressor(loss="absolute_error", n_estimators=200,
                                           learning_rate=0.05, max_depth=2,
                                           random_state=0).fit(
    fit_x.reshape(-1, 1), corrupted_y).predict(held_x.reshape(-1, 1))
their_squared = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2,
                                          random_state=0).fit(
    fit_x.reshape(-1, 1), corrupted_y).predict(held_x.reshape(-1, 1))

print(pd.DataFrame([
    {"what the tree is fitted to": "sign of residual (mine)", "leaf values": "tree's own",
     "held-out RMSE": held_out_rmse(plain_sign)},
    {"what the tree is fitted to": "sign of residual (mine)", "leaf values": "median of residuals",
     "held-out RMSE": held_out_rmse(median_sign)},
    {"what the tree is fitted to": "sign of residual (sklearn)", "leaf values": "median of residuals",
     "held-out RMSE": held_out_rmse(their_absolute)},
    {"what the tree is fitted to": "residual itself (sklearn)", "leaf values": "mean of residuals",
     "held-out RMSE": held_out_rmse(their_squared)},
]).to_string(index=False, float_format=lambda v: "%.4f" % v))

**Every sign-based variant lands near 1.62-1.65 on data that takes squared error to 5.4986.** Fitting the
sign recovers essentially all of the damage; the differences among the three robust rows are small enough
that this data cannot rank them.

**None of the three robust versions matches another exactly, and the reason is the leaf values.** Fitting a
tree to +1 and -1 makes each leaf predict the *proportion* of positive residuals in it, rescaled - a number
between -1 and 1 that says which way to move but not how far. sklearn keeps the tree's splits and then
replaces each leaf's value with the **median** of the residuals that landed there, because the median is
what minimises absolute error within a leaf exactly as the mean minimises squared error. That is a line
search, and it is 05-06's idea of a step size chosen rather than guessed.

**And here the measurement refuses to reward the refinement.** Adding medians moved my answer from 1.6162
to 1.6500 - away from sklearn's 1.6292, not towards it, and slightly worse than the crude version. Do not
read that as the refinement being wrong: with 200 rows and a step of 0.05 the three numbers are within a
few hundredths of each other, which is well inside what a different split of this data would move them.
**What the table does establish is the thing worth remembering: the 5.4986 collapses to roughly 1.63 the
moment the tree is handed a sign instead of a magnitude, and every later detail is a rounding error next
to that.**

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.4))
grid = np.linspace(-4, 4, 400)

# what each loss asks the tree to fit, as a function of the residual
resid_axis = np.linspace(-6, 6, 400)
left.plot(resid_axis, resid_axis, lw=2.2, color="#c0392b", label="squared error: fit $r$")
left.plot(resid_axis, np.sign(resid_axis), lw=2.2, color="#1f6f4a",
          label=r"absolute error: fit $\mathrm{sign}(r)$")
left.axhline(0, color="#999999", lw=0.8)
left.axvline(0, color="#999999", lw=0.8)
left.scatter([5.5], [5.5], s=70, color="#c0392b", zorder=5)
left.scatter([5.5], [1.0], s=70, color="#1f6f4a", zorder=5)
left.annotate("a row out by 5.5 shouts\nunder squared error", xy=(5.5, 5.5), xytext=(-2.4, 4.6),
              fontsize=9, color="#c0392b", ha="center",
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
              arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2))
left.annotate("...and whispers 'up'\nunder absolute error", xy=(5.5, 1.0), xytext=(-1.0, -4.2),
              fontsize=9, color="#1f6f4a", ha="center",
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
              arrowprops=dict(arrowstyle="->", color="#1f6f4a", lw=1.2))
left.set_xlabel("residual $r = y - F$")
left.set_ylabel("target handed to the next tree")
left.set_title("the negative gradient is what the tree sees")
left.legend(loc="lower right", fontsize=8.5)

# the two fitted curves on the corrupted data
intact = np.setdiff1d(np.arange(len(corrupted_y)), damaged)
right.scatter(fit_x[intact], corrupted_y[intact], s=12, color="#b0b0b0", label="fitting rows")
right.scatter(fit_x[damaged], np.full(len(damaged), 13.0), s=110, marker="^", color="#c0392b",
              zorder=5, label="5 corrupted rows, drawn at the top (really at +40)")
right.plot(grid, GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2,
                                           random_state=0).fit(
    fit_x.reshape(-1, 1), corrupted_y).predict(grid.reshape(-1, 1)),
    lw=2.4, color="#c0392b", label="squared error")
right.plot(grid, true_curve(grid), lw=2.0, color="#333333", ls="--", label="true curve")
on_fit = np.full(len(corrupted_y), np.median(corrupted_y))
on_grid = np.full(len(grid), np.median(corrupted_y))
for _ in range(200):
    tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(
        fit_x.reshape(-1, 1), np.sign(corrupted_y - on_fit))
    on_fit = on_fit + 0.05 * tree.predict(fit_x.reshape(-1, 1))
    on_grid = on_grid + 0.05 * tree.predict(grid.reshape(-1, 1))
right.plot(grid, on_grid, lw=2.4, color="#1f6f4a", label="absolute error (sign)")
right.set_ylim(-8.0, 14.5)
right.set_xlabel("x")
right.set_ylabel("y")
right.set_title("what the five bad rows do to each fit")
right.legend(loc="lower right", fontsize=8)

fig.suptitle("E20 - changing the loss changes only what the tree is handed", y=1.01)
fig.tight_layout()
plt.show()

**Left panel: the entire difference between the two algorithms, in one picture.** Squared error hands the
tree a target that grows without limit as a row gets worse, so a row out by 40 is forty times as loud as a
row out by 1. Absolute error flattens that to +1 or -1: **every wrong row gets one vote, regardless of how
wrong it is.** Robustness is not a special mechanism bolted on; it is what a bounded gradient does.

**Right panel: the consequence.** The red curve throws up a narrow spike at each corrupted row - it is
chasing targets that sit at +40, far above the window, and a depth-2 tree can only reach them by isolating
those rows. The green curve runs straight past all five and tracks the true dashed curve, because under
absolute error the bad rows outvoted nobody. **Look at where the spikes are and where the triangles are:
they line up exactly.**

**When to reach for it:** corrupted targets, heavy tails, or any setting where the extreme rows are
genuinely errors rather than genuinely extreme. **When not to:** when the large values are real and
matter - a fraud amount, a peak load - because absolute error will deliberately underfit exactly those
rows. 05-04's question, "what does a mistake cost?", is the one that decides it, and it now decides the
*training* loss and not just the report.

### E21 · A linear base learner, at rate 1, is ordinary least squares

Start at the mean, fit one linear model to the residuals, add all of it. The claim is that the result is
the OLS line - and that the second round then does nothing at all.

In [ ]:
straight = LinearRegression().fit(fit_x.reshape(-1, 1), fit_y)

on_fit = np.full(len(fit_y), fit_y.mean())
rows = []
for step in range(1, 4):
    residual = fit_y - on_fit
    learner = LinearRegression().fit(fit_x.reshape(-1, 1), residual)
    on_fit = on_fit + 1.0 * learner.predict(fit_x.reshape(-1, 1))
    rows.append({"round": step,
                 "slope this round added": round(float(learner.coef_[0]), 6) + 0.0,
                 "intercept this round added": round(float(learner.intercept_), 6) + 0.0,
                 "max |boosted - OLS|": round(float(np.abs(
                     on_fit - straight.predict(fit_x.reshape(-1, 1))).max()), 12) + 0.0})
print(pd.DataFrame(rows).to_string(index=False))
print("\nOLS directly:  slope %.6f  intercept %.6f"
      % (straight.coef_[0], straight.intercept_))
print("OLS held-out RMSE %.4f, against about 1.52 for a boosted ensemble on this data"
      % held_out_rmse(straight.predict(held_x.reshape(-1, 1))))

**Round 1 adds slope 0.281788, which is exactly the OLS slope, and the two models then agree to every
decimal place printed. Rounds 2 and 3 add zero.**

**Why round 1 lands on OLS.** The starting prediction is the mean, so the residual is `y - mean(y)`.
Regressing that on `x` gives the OLS slope unchanged - subtracting a constant from the target moves the
intercept and leaves the slope alone - and adding the fitted line to the mean reassembles precisely
`mean(y) - b*mean(x) + b*x`, which is the OLS line.

**Why round 2 adds nothing.** OLS residuals are, by construction, orthogonal to every column in the
design matrix. Fitting a linear model to something orthogonal to `x` returns slope zero. **The learner has
nothing left it is capable of seeing.**

> **This is what "weak learner" means, and it is a statement about capacity, not about quality.** The base
> learner has to be *unable* to solve the problem in one step - otherwise it does, and the sequence ends
> after one round with no ensemble at all.

**Two consequences worth carrying.**

1. **Boosting a strong learner is pointless, not dangerous.** With a linear learner the process converges
   after one round; with a full-depth tree it converges after one round too, having memorised the training
   data. In both cases the ensemble adds nothing, which is why depth 2 to 4 is the working range.
2. **The gains come from repeated small corrections in directions each individual learner cannot take.**
   A stump can only see one variable at a time; sixty stumps in sequence can describe a curve, because
   each is fitted to what the previous fifty-nine could not represent. That is why boosting turns a
   deliberately incapable model into a capable one - and why making the base learner capable destroys the
   mechanism.

**A last sanity check on the numbers:** the OLS line scores 2.5988 held-out on this data, against roughly
1.52 for a properly boosted ensemble. One round of a strong learner is not merely inelegant - on a curved
problem it is a much worse model.